In [1]:
%pip install -q -U langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-classic faiss-cpu pypdf youtube-transcript-api beautifulsoup4 requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q -U sentence-transformers langchain-huggingface langchain-ollama faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import tempfile
import requests

from urllib.parse import urlparse, parse_qs
from pypdf import PdfReader
from bs4 import BeautifulSoup
from youtube_transcript_api import YouTubeTranscriptApi

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import FAISS

print("All libraries imported successfully!")

All libraries imported successfully!


C:\Users\sruth\AppData\Local\Temp\ipykernel_35680\3844108498.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model initialized!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model initialized!


In [5]:
vector = embeddings.embed_query(
    "This is my RAG project."
)

print("Embedding created successfully!")
print("Vector length:", len(vector))
print("First 5 values:", vector[:5])

Embedding created successfully!
Vector length: 384
First 5 values: [-0.059148866683244705, 0.13866238296031952, 0.07090914994478226, 0.005306579172611237, -0.08114998787641525]


In [6]:
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

print("Local LLM initialized!")

Local LLM initialized!


In [7]:
response = llm.invoke(
    "Explain RAG in two simple sentences."
)

print(response.content)

RAG is a system used to categorize and prioritize tasks based on their urgency and importance. It stands for "Must-Do", "Should-Do", and "Can-Do", helping individuals and teams focus on the most critical tasks first.


In [8]:
def extract_youtube_transcript(url):
    parsed_url = urlparse(url)
    hostname = parsed_url.hostname

    if not hostname:
        raise ValueError("Invalid YouTube URL.")

    if hostname == "youtu.be":
        video_id = parsed_url.path.strip("/").split("/")[0]

    elif hostname == "youtube.com" or hostname.endswith(".youtube.com"):

        if parsed_url.path == "/watch":
            video_id = parse_qs(
                parsed_url.query
            ).get("v", [None])[0]

        elif parsed_url.path.startswith(("/embed/", "/shorts/")):
            parts = parsed_url.path.strip("/").split("/")
            video_id = parts[1] if len(parts) > 1 else None

        else:
            video_id = None

    else:
        raise ValueError("Please provide a valid YouTube URL.")

    if not video_id:
        raise ValueError("Could not find the YouTube video ID.")

    transcript = YouTubeTranscriptApi().fetch(video_id)

    text = " ".join(
        snippet.text for snippet in transcript
    )

    if not text.strip():
        raise ValueError("The transcript is empty.")

    return [
        Document(
            page_content=text,
            metadata={
                "source": url,
                "type": "youtube",
                "video_id": video_id
            }
        )
    ]

In [9]:
def extract_pdf(pdf_path_or_url):
    temp_path = None

    try:
        if pdf_path_or_url.lower().startswith(("http://", "https://")):

            response = requests.get(
                pdf_path_or_url,
                timeout=30
            )

            response.raise_for_status()

            if not response.content.startswith(b"%PDF"):
                raise ValueError(
                    "The URL did not return a valid PDF file."
                )

            with tempfile.NamedTemporaryFile(
                suffix=".pdf",
                delete=False
            ) as temp_file:

                temp_file.write(response.content)
                temp_path = temp_file.name

            file_path = temp_path

        else:
            file_path = pdf_path_or_url

        reader = PdfReader(file_path)
        documents = []

        for page_number, page in enumerate(
            reader.pages,
            start=1
        ):

            text = page.extract_text() or ""

            if text.strip():
                documents.append(
                    Document(
                        page_content=text,
                        metadata={
                            "source": pdf_path_or_url,
                            "type": "pdf",
                            "page": page_number
                        }
                    )
                )

        if not documents:
            raise ValueError(
                "No readable text found. The PDF may be scanned."
            )

        return documents

    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

In [10]:
documents = extract_pdf("sruthi_vaddadhi.cv_new.pdf")

print("PDF extracted successfully!")
print("Total pages with readable text:", len(documents))

print("\nFirst page preview:")
print(documents[0].page_content[:500])

PDF extracted successfully!
Total pages with readable text: 2

First page preview:
  
      SRUTHI VADDADHI  
 B. Tech – Computer Science and Engineering  
 Roll number: 24CS8007  
 Gender: Female                                                                 
 Date of Birth:16/06/2007  
 Category: SC  
 E-mail: sruthivaddadi745@gmail.com  
Contact:8978116861   
Address: Visakhapatnam, Andhra Pradesh     
                                       LinkedIn  GitHub   LeetCode  
  
Educational Qualification               
● Graduation  
SEM  1  2  3  4  
SGPA  7.91  8.00  7.62  7


In [11]:
def extract_webpage(url):
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    response.raise_for_status()

    content_type = response.headers.get(
        "Content-Type", ""
    ).lower()

    if "text/html" not in content_type:
        raise ValueError(
            "The URL is not an HTML webpage."
        )

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    for element in soup(
        ["script", "style", "nav", "footer", "header", "noscript"]
    ):
        element.decompose()

    text = soup.get_text(
        separator=" ",
        strip=True
    )

    if not text:
        raise ValueError(
            "No readable text found on the webpage."
        )

    return [
        Document(
            page_content=text,
            metadata={
                "source": url,
                "type": "webpage"
            }
        )
    ]

In [12]:
def load_source(source):
    source = source.strip()

    if not source:
        raise ValueError("Please enter a source.")

    # Local PDF
    if os.path.isfile(source):

        if source.lower().endswith(".pdf"):
            return extract_pdf(source)

        raise ValueError(
            "Only local PDF files are supported."
        )

    # URL validation
    if not source.lower().startswith(("http://", "https://")):
        raise ValueError(
            "Enter a valid URL or a PDF file path."
        )

    parsed = urlparse(source)
    hostname = parsed.hostname

    if not hostname:
        raise ValueError("Invalid URL.")

    # YouTube URL
    is_youtube = (
        hostname == "youtube.com"
        or hostname.endswith(".youtube.com")
        or hostname == "youtu.be"
        or hostname.endswith(".youtu.be")
    )

    if is_youtube:
        return extract_youtube_transcript(source)

    # Direct PDF URL
    if parsed.path.lower().endswith(".pdf"):
        return extract_pdf(source)

    # Webpage URL
    return extract_webpage(source)

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True
)


def split_documents(documents):
    chunks = text_splitter.split_documents(documents)

    print("Total documents:", len(documents))
    print("Total chunks:", len(chunks))

    return chunks

In [14]:
chunks = split_documents(documents)

print("\nFirst chunk preview:")
print(chunks[0].page_content[:500])

Total documents: 2
Total chunks: 7

First chunk preview:
SRUTHI VADDADHI  
 B. Tech – Computer Science and Engineering  
 Roll number: 24CS8007  
 Gender: Female                                                                 
 Date of Birth:16/06/2007  
 Category: SC  
 E-mail: sruthivaddadi745@gmail.com  
Contact:8978116861   
Address: Visakhapatnam, Andhra Pradesh     
                                       LinkedIn  GitHub   LeetCode  
  
Educational Qualification               
● Graduation  
SEM  1  2  3  4  
SGPA  7.91  8.00  7.62  7.55  
CGP


In [15]:
def create_vector_store(chunks):
    if not chunks:
        raise ValueError("No chunks available.")

    vector_store = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    print("FAISS vector store created successfully!")

    return vector_store

In [16]:
vector_store = create_vector_store(chunks)

FAISS vector store created successfully!


In [26]:

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a precise and professional question-answering assistant.

Your task is to answer the user's question strictly using the provided context.

Follow these rules carefully:
1. Answer only what the user asks. Do not include unrelated information, additional suggestions, or unnecessary explanations.
2. Write the answer as one clear, concise, and grammatically correct paragraph.
3. Start directly with the answer. Do not include introductions such as "Based on the provided context" or "According to the document."
4. Do not use bullet points, numbered lists, headings, or unnecessary formatting.
5. Do not repeat the question in your answer.
6. Do not invent information or use knowledge outside the provided context.
7. If the context does not contain enough information to answer the question, respond exactly with: "I could not find the answer in the provided content."
8. Keep the answer relevant, accurate, and professional.

Context:
{context}"""
    ),
    (
        "human",
        "{question}"
    )
])


In [27]:
def build_rag_chain(vector_store):

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )

    rag_chain = (
        RunnableParallel({
            "context": retriever,
            "question": RunnablePassthrough()
        })
        | RunnableParallel({
            "answer": (
                prompt
                | llm
                | StrOutputParser()
            ),
            "context": lambda x: x["context"],
            "question": lambda x: x["question"]
        })
    )

    return rag_chain

In [28]:
rag_chain = build_rag_chain(vector_store)

print("RAG chain created successfully!")

RAG chain created successfully!


In [29]:
def ask_question(question):

    if not question.strip():
        print("Please enter a question.")
        return

    result = rag_chain.invoke(question)

    print("\nQUESTION:")
    print(result["question"])

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")

    for i, doc in enumerate(
        result["context"],
        start=1
    ):

        print(f"\nSource {i}")

        print(
            "Type:",
            doc.metadata.get("type", "Unknown")
        )

        print(
            "Location:",
            doc.metadata.get("source", "Unknown")
        )

        if "page" in doc.metadata:
            print(
                "Page:",
                doc.metadata["page"]
            )

        print(
            "Text:",
            doc.page_content[:300]
        )

In [34]:
source = input(
    "Enter a YouTube URL, webpage URL, or PDF path: "
).strip()

try:
    documents = load_source(source)

    print("\nContent extracted successfully!")

    chunks = split_documents(documents)

    vector_store = create_vector_store(chunks)

    rag_chain = build_rag_chain(vector_store)

    print("\nYour RAG system is ready!")

except Exception as e:
    print("\nError:", str(e))

Enter a YouTube URL, webpage URL, or PDF path:  sruthi_vaddadhi.cv_new.pdf



Content extracted successfully!
Total documents: 2
Total chunks: 7
FAISS vector store created successfully!

Your RAG system is ready!


In [35]:
ask_question(
    "What technical skills are mentioned in my resume?"
)


QUESTION:
What technical skills are mentioned in my resume?

ANSWER:
Technical skills mentioned in your resume are:

1. C
2. C++
3. HTML
4. CSS
5. JavaScript
6. Python
7. Tailwind CSS
8. Bootstrap
9. Node.js
10. React js
11. Numpy
12. Pandas
13. LangChain
14. Git
15. Excel
16. Pandas
17. Visual Studio Code
18. LeetCode
19. Vercel
20. Render
21. MySQL
22. MongoDB
23. Firebase
24. Gemini API
25. OpenAI API
26. FAISS
27. REST APIs
28. JWT Authentication

SOURCES:

Source 1
Type: pdf
Location: sruthi_vaddadhi.cv_new.pdf
Page: 1
Text: ● Others                                 :DSA(C++), OOP , Problem Solving, Operating systems, Competitive Programming.  
● Non-Technical Skills         : Teamwork, Communication, Problem Solving, Time Management, Adaptability.

Source 2
Type: pdf
Location: sruthi_vaddadhi.cv_new.pdf
Page: 1
Text: ● Solved 300+ coding problems and actively participated in coding contests with a maximum rating of 1419 across 
platforms such as Leetcode .  
● Selected among top 

In [23]:
ask_question(
    "What technologies did I use in my projects?"
)


QUESTION:
What technologies did I use in my projects?

ANSWER:
Based on the provided context, here are the technologies you used in your projects:

**Web Development:**

* React.js
* Node.js
* Express.js
* MongoDB
* JWT Authentication
* Cloudinary
* Render
* Vercel
* GitHub

**Blockchain and Distributed Systems:**

* Blockchain-based Blue Carbon Registry & MRV System (React, HTML, CSS, JavaScript, Node.js, Django, MongoDB, Ethereum, Solidity, IPFS)
* E-Commerce (React.js, Node.js, Express.js, MongoDB, JWT Authentication, Cloudinary, Render, Vercel, GitHub)

**Data Science and AI:**

* Gemini API
* OpenAI API
* FAISS
* REST APIs
* JWT Authentication

**Database:**

* MySQL
* MongoDB

**Programming Languages:**

* C
* C++
* HTML
* CSS
* JavaScript
* Python

**Frameworks and Libraries:**

* Tailwind CSS
* Bootstrap
* LangChain
* Numpy
* Pandas
* LangChain (parallel retrieval to prompt injection to generation)
* GPT-4o-mini (OpenAI API)

**Other:**

* Git
* Excel
* Pandas
* Visual Studio 

In [24]:
ask_question(
    "go through the live demos of my project explain what are listed"
)


QUESTION:
go through the live demos of my project explain what are listed

ANSWER:
I'll go through the live demos of your project and explain what is listed.

**Forever-Ecommerce**

The live demo is for the "Forever-Ecommerce" project, which is a full-stack e-commerce platform built using React.js, Node.js, Express.js, MongoDB, JWT Authentication, Cloudinary, Render, Vercel, and GitHub.

Here are the features listed in the live demo:

* **Admin Dashboard**: A secure REST API for managing inventory, order status updates, and cloudinary-based image uploads.
* **Order Management**: Secure REST APIs for managing orders, including order status updates and payment processing.
* **Inventory Management**: A feature branch workflow for managing inventory, including tracking stock levels and updating inventory in real-time.
* **Cloudinary-based Image Uploads**: Secure REST APIs for uploading images to Cloudinary, with secure authentication and authorization.
* **Live Demo**: A live demo of the 